In [ ]:
#2026.03.22 최종 df버전
#..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv로 저장했습니다.
#완료 - 20260322_11-15-45

In [ ]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, expand_window
import pandas as pd
import numpy as np
import gc
from pandas.api.types import is_categorical_dtype
import re
from pathlib import Path

import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.v_affix_attach import v_affix_attach, add_v_no_by_merge_and_fallback
from stats.en_no_fix import fix_en_number_with_merge, make_en_no_sub #EN_No 수정
from stats.filtering import apply_filters, FilterValue, has_value, _topn_values


In [3]:
from pathlib import Path
from datetime import datetime

#대상 파일 읽어오기.
CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv")
df = pd.read_csv(CSV_PATH)

stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
print(f"*** 파일 읽기 완료: {CSV_PATH} ({len(df):,}행), {stamp} ***")

*** 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv (12,973,652행), 20260307_132546 ***


In [5]:
df.columns.all

<bound method Index.all of Index(['ID', 'category', 'file_id', 'docu_id', 'sen_id', 'word_id2',
       'rev_word_id2', 'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0',
       'V_label_0', 'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'vx_len',
       'Next_VX_No', 'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order',
       'V_word_id', 'f_word_id', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'docu_lenth', 'dominant_EN_No', 'dominant_count',
       'dominant_ratio', 'quote_type', 'quote_end_offset'],
      dtype='object')>

In [7]:
df_word = df.copy()

In [14]:
#문장 파일 읽기
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.5_sen.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)
#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
print(f"*** 파일 읽기 완료: {SEN_CSV} 등 ({len(df_sen):,}행), {stamp} ***")

*** 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.5_sen.csv 등 (1,048,127행), 20260307_13-50-52 ***


In [47]:
df_sen.columns.all

<bound method Index.all of Index(['sen_id', 'file_id', 'file_name', 'docu_id', 'docu_no', 'sen_num',
       'sentence_form', 'speaker', 'sen_len', 'quote_count', 'quote_positions',
       'quote_type', 'quote_group_id', 'quote_span_len', 'long_quote',
       'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok', 'span_type',
       'passage_type', 'review_flag', 'review_note', 'quote_type_fix',
       'modified_at', 'quote_end_offset'],
      dtype='object')>

In [23]:
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.6_sen.csv"     
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

df_sen.to_csv(SEN_CSV, index=False, encoding= "utf-8")
print(f"{SEN_CSV}로 저장했습니다. {stamp}")


..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.6_sen.csv로 저장했습니다. 20260307_140549


In [24]:
df_docu.columns

Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'docu_lenth', 'dominant_EN_No', 'dominant_count',
       'dominant_ratio'],
      dtype='object')

In [25]:
keep_cols = [
    'docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4',
]

keep_cols = [c for c in keep_cols if c in df_docu.columns]
df_docu = df_docu[keep_cols].copy()

In [ ]:
#후처리1 - 문장 끝 표시, 문장 끝 기준 역방향 번호, 문장 끝 V 여부
# 'sent_end', 'rev_word_id2', 'sent_end_V' 컬럼 추가
#sent_end: 문장 끝 표시 (True/False)
#rev_word_id2: 문장 끝 기준 역방향 번호 (문장 끝이 0, 그 앞이 -1, -2 ... 이런 식으로 번호가 매겨지는 것)
#sent_end_V: 문장 끝 V 여부 (True/False, vx0_order == -1인 V 중 가장 뒤에 있는 것 이후의 V는 모두 True)

import pandas as pd
import sys
from pathlib import Path
from stats.filtering import has_value

# 0. 정렬 보장
df = (
    df
    .sort_values(["docu_id", "sen_id", "word_id2"])  # word_id는 문장 내부 순서
    .reset_index(drop=True)
)

# 1) sentence-end (always one per sen_id)
last_row_idx = df.groupby("sen_id").tail(1).index

df["sent_end"] = False
df.loc[last_row_idx, "sent_end"] = True

#2) 문장 끝 기준 역방향 번호
max_word_id = df.groupby("sen_id")["word_id2"].transform("max") # 문장별 최대 word_id2
df["rev_word_id2"] = df["word_id2"] - max_word_id # 문장 끝 기준 역방향 번호

#3) 문장 끝 V 여부 판단
#  1. V 여부 마스크
v_mask = has_value(df["V_label"]) | has_value(df["EN_label"])

# 2. 문장별 기준점 계산
#    (vx0_order == -1 인 V 중 가장 뒤에 있는 것)

core_mask = v_mask & (df["vx0_order"] == -1)

# 문장 내부에서 마지막 core V의 위치(문장 내 순번) 구하기
df["word_pos"] = df.groupby("sen_id").cumcount()

last_core_pos = (
    df.loc[core_mask]
    .groupby("sen_id")["word_pos"]
    .max()
)

# 3. 기준점 이후 V 전부 True 처리
df["sent_end_V"] = False

# 기준점 매핑
df = df.merge(
    last_core_pos.rename("last_core_pos"),
    on="sen_id",
    how="left"
)

df["sent_end_V"] = (
    v_mask &
    (df["word_pos"] >= df["last_core_pos"])
)

# 4. 보조 컬럼 제거
df.drop(columns=["word_pos", "last_core_pos"], inplace=True)


In [ ]:
# 문장 끝 V가 인용구 안에 있는지 여부 표시
# 원칙:
# 1) 먼저 "문장 끝 core V(vx0_order == -1)" 가 인용구 안인지 판정
# 2) 그 core V가 인용구 안인 문장에서는,
#    sent_end_V와 동일하게 그 기준점 이후의 V 전부를 True 처리

# ----------------------------------
# 0. quote_type, quote_end_offset 병합
# ----------------------------------

quote_info_cols = ["quote_type", "quote_end_offset"] # 필요한 컬럼명 리스트로 정의
rest_cols = [col for col in df.columns if col not in quote_info_cols] # 원본 df에서 quote_info_cols를 제외한 나머지 컬럼명 리스트
df = df[rest_cols].copy() # quote_info_cols는 뒤에서 다시 병합할 예정이므로 일단 제거

df = df.merge(
    df_sen[["sen_id"]+ quote_info_cols],
    on="sen_id",
    how="left",
    validate="many_to_one"
)

# -------------------------------------------------
# 1. 기본 준비
# -------------------------------------------------
v_mask = has_value(df["V_label"]) | has_value(df["EN_label"])
core_mask = v_mask & (df["vx0_order"] == -1)

df["word_pos"] = df.groupby("sen_id").cumcount()

# -------------------------------------------------
# 2. core V가 인용구 안인지 먼저 판정
# -------------------------------------------------
df["core_V_in_quote"] = False

# full / middle
df.loc[
    core_mask
    & df["quote_type"].isin(["full_quote", "quote_middle"]),
    "core_V_in_quote"
] = True

# partial / close
df.loc[
    core_mask
    & df["quote_type"].isin(["partial_quote", "quote_close"])
    & (df["rev_word_id2"] < df["quote_end_offset"]),
    "core_V_in_quote"
] = True

# open
df.loc[
    core_mask
    & (df["quote_type"] == "quote_open")
    & (df["rev_word_id2"] > df["quote_end_offset"]),
    "core_V_in_quote"
] = True

# -------------------------------------------------
# 3. 문장별 마지막 core V 위치와,
#    그 core V가 인용구 안인지 여부를 문장 단위로 계산
# -------------------------------------------------
last_core_pos = (
    df.loc[core_mask]
      .groupby("sen_id")["word_pos"]
      .max()
      .rename("last_core_pos")
)

last_core_in_quote = (
    df.loc[core_mask & df["core_V_in_quote"]]
      .groupby("sen_id")
      .size()
      .gt(0)
      .rename("last_core_in_quote")
)

df = df.merge(last_core_pos, on="sen_id", how="left")
df = df.merge(last_core_in_quote, on="sen_id", how="left")

df["last_core_in_quote"] = df["last_core_in_quote"].fillna(False)

# -------------------------------------------------
# 4. sent_end_V와 같은 범위로 확장
#    = 마지막 core V가 인용구 안인 문장에서
#      기준점 이후 V 전부 True
# -------------------------------------------------
df["sent_end_V_in_quote"] = (
    v_mask
    & (df["word_pos"] >= df["last_core_pos"])
    & df["last_core_in_quote"]
)

# -------------------------------------------------
# 5. 보조 컬럼 제거
# -------------------------------------------------
df.drop(
    columns=["word_pos", "core_V_in_quote", "last_core_pos", "last_core_in_quote"],
    inplace=True
)


C:\Users\yu2hy\AppData\Local\Temp\ipykernel_11480\2715309149.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["last_core_in_quote"] = df["last_core_in_quote"].fillna(False)


In [28]:
df.columns

Index(['ID', 'category', 'file_id', 'docu_id', 'sen_id', 'word_id2',
       'rev_word_id2', 'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0',
       'V_label_0', 'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'vx_len',
       'Next_VX_No', 'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order',
       'V_word_id', 'f_word_id', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'docu_lenth', 'dominant_EN_No', 'dominant_count',
       'dominant_ratio', 'quote_type', 'quote_end_offset'],
      dtype='object')

In [19]:
df = df.rename(columns={"Next_VX_No_form": "Next_VX_form"})


In [25]:
keep_cols = ['ID', 'category', 'file_id', 'docu_id', 'sen_id', 'word_id2',
       'rev_word_id2', 'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0',
       'V_label_0', 'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'vx_len',
       'Next_VX_No', 'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order',
       'V_word_id', 'f_word_id', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'quote_type', 'quote_end_offset',
       'docu_lenth', 'dominant_EN_No', 'dominant_count', 'dominant_ratio'
]


keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

In [ ]:
### VX가 없는 경우, word_id2로 V_word_id, f_word_id 채우기

mask = (df_word["V_form"].notna()) & (df_word["V_word_id"] == -1)
df_word.loc[mask, "V_word_id"] = df_word.loc[mask, "word_id2"]
mask = (df_word["EN_form"].notna()) & (df_word["f_word_id"] == -1)
df_word.loc[mask, "f_word_id"] = df_word.loc[mask, "word_id2"]

In [53]:
output_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv"
df.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
print(f"완료 - {stamp}")

..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv로 저장했습니다.
완료 - 20260307_11-25-55


In [33]:
df_docu.columns.all

<bound method Index.all of Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'docu_lenth', 'dominant_EN_No', 'dominant_count',
       'dominant_ratio'],
      dtype='object')>

In [ ]:
keep_cols = ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2','파일제목', '저자',
       '출판사', '출판연도', 'head', 
       '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4'
]

keep_cols = [c for c in keep_cols if c in df_docu.columns]
df_docu = df_docu[keep_cols].copy()

In [ ]:
#docu_id 기준으로 df에서 docu_lenth, dominant_EN_No, dominant_count, dominant_ratio 정보 추출하여 df_docu에 병합
df_meta = df[['docu_id', 'docu_lenth', 'dominant_EN_No', 'dominant_count', 'dominant_ratio']].drop_duplicates('docu_id')

df_docu = df_docu.merge(df_meta, on='docu_id', how='left')

In [15]:
#----
# df_sen에 has_E, final_E 컬럼 추가, last_EN_in_quote 
#---
# #has_E: 해당 어절에 EN이 있는지 여부, final_E: 문장 내 마지막 EN의 EN_No 또는 EN_label (필요한 경우 둘 다)
import pandas as pd
from datetime import datetime

# df_word copy
df_word = df_word.copy()

# 1) whether each eojeol has EN
df_word["has_E"] = (
    df_word["EN_No"].notna() &
    df_word["EN_label"].notna()
)

# 2) sentence-level: whether the sentence contains any EN
sen_has_E = (
    df_word.groupby("sen_id", as_index=False)["has_E"]
    .any()
    .rename(columns={"has_E": "sent_has_E"})
)

# 3) keep only rows with EN, then sort by eojeol order
df_en = df_word[df_word["EN_No"].notna()].copy()
df_en = df_en.sort_values(["sen_id", "word_id2"])

# 4) get the last EN row in each sentence
last_en_info = (
    df_en.groupby("sen_id", as_index=False)
    .tail(1)[["sen_id", "EN_No", "EN_label", "sent_end_V_in_quote"]]
    .rename(columns={
        "EN_No": "last_EN_No",
        "EN_label": "last_EN_label",
        "sent_end_V_in_quote": "last_EN_in_quote"
    })
)

# 5) merge into df_sen
df_sen = df_sen.merge(sen_has_E, on="sen_id", how="left")
df_sen = df_sen.merge(last_en_info, on="sen_id", how="left")

# 6) fill missing values
df_sen["sent_has_E"] = df_sen["sent_has_E"].fillna(False)
df_sen["last_EN_in_quote"] = df_sen["last_EN_in_quote"].fillna(False)

stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
print(f"완료 - {stamp}")

완료 - 20260307_135221


C:\Users\yu2hy\AppData\Local\Temp\ipykernel_69880\1376757328.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sen["sent_has_E"] = df_sen["sent_has_E"].fillna(False)
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_69880\1376757328.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sen["last_EN_in_quote"] = df_sen["last_EN_in_quote"].fillna(False)


In [ ]:
df_sen.to_csv(DOCU_CSV, index=False, encoding= "utf-8")
print(f"{DOCU_CSV}로 저장했습니다.")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"완료 - {stamp}")

In [22]:
df_sen.columns

Index(['sen_id', 'file_id', 'file_name', 'docu_id', 'docu_no', 'sen_num',
       'sentence_form', 'speaker', 'sen_len', 'quote_count', 'quote_positions',
       'quote_type', 'quote_group_id', 'quote_span_len', 'long_quote',
       'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok', 'span_type',
       'passage_type', 'review_flag', 'review_note', 'quote_type_fix',
       'modified_at', 'quote_end_offset', 'sent_has_E', 'last_EN_No',
       'last_EN_label', 'last_EN_in_quote'],
      dtype='object')

In [18]:
#문서별 dominant EN_No 정보 추가 함수
import pandas as pd
import numpy as np
from typing import Optional, List

def add_docu_sentence_en_info(
    df_docu: pd.DataFrame,
    df_sen: pd.DataFrame,
    *,
    group_columns: Optional[List[str]] = None,
    en_no_col: str = "last_EN_No",
    en_label_col: str = "last_EN_label",
    has_e_col: str = "sent_has_E",
    in_quote_col: str = "last_EN_in_quote",
    dominance_threshold: Optional[float] = None,   # optional
) -> pd.DataFrame:
    """
    Add document-level sentence statistics and dominant EN info to df_docu.

    Parameters
    ----------
    df_docu : document-level dataframe
    df_sen : sentence-level dataframe
    group_columns : grouping keys, default ["docu_id"]
    en_no_col : sentence-final EN_No column in df_sen
    en_label_col : sentence-final EN_label column in df_sen
    has_e_col : whether the sentence contains EN
    in_quote_col : whether the sentence-final EN is in quote
    dominance_threshold : optional threshold for separate flag columns

    Returns
    -------
    df_docu with added columns
    """
    if group_columns is None:
        group_columns = ["docu_id"]

    df_docu = df_docu.copy()
    df_sen = df_sen.copy()

    # ----------------------------
    # 0) basic cleanup
    # ----------------------------
    df_sen[has_e_col] = df_sen[has_e_col].fillna(False).astype(bool)
    df_sen[in_quote_col] = df_sen[in_quote_col].fillna(False).astype(bool)
    df_sen[en_no_col] = pd.to_numeric(df_sen[en_no_col], errors="coerce")

    # ----------------------------
    # 1) document-level sentence counts
    # ----------------------------
    sen_count = (
        df_sen.groupby(group_columns)
        .size()
        .reset_index(name="sen_count")
    )

    sen_count_has_E = (
        df_sen[df_sen[has_e_col]]
        .groupby(group_columns)
        .size()
        .reset_index(name="sen_count_has_E")
    )

    sen_count_not_quote = (
        df_sen[~df_sen[in_quote_col]]
        .groupby(group_columns)
        .size()
        .reset_index(name="sen_count_not_quote")
    )

    sen_count_has_E_not_quote = (
        df_sen[df_sen[has_e_col] & ~df_sen[in_quote_col]]
        .groupby(group_columns)
        .size()
        .reset_index(name="sen_count_has_E_not_quote")
    )

    # ----------------------------
    # 2) dominant EN among non-quote sentences
    #    (recommended: only sentences that actually have EN)
    # ----------------------------
    target = df_sen[df_sen[has_e_col] & ~df_sen[in_quote_col]].copy()

    if target.empty:
        dominant_info = df_docu[group_columns].drop_duplicates().copy()
        dominant_info["dominant_EN_No"] = pd.NA
        dominant_info["dominant_EN_label"] = pd.NA
        dominant_info["dominant_count"] = pd.NA
        dominant_info["dominant_ratio"] = pd.NA

        if dominance_threshold is not None:
            dominant_info["dominant_over_threshold"] = pd.NA
    else:
        # frequency by EN_No + EN_label
        freq = (
            target.groupby(group_columns + [en_no_col, en_label_col], dropna=False)
            .size()
            .reset_index(name="count")
        )

        total = (
            target.groupby(group_columns)
            .size()
            .reset_index(name="base_count_not_quote")
        )

        freq = freq.merge(total, on=group_columns, how="left")

        # choose the most frequent EN per document
        # if tied, first one after sorting by count desc, EN_No asc
        freq = freq.sort_values(group_columns + ["count", en_no_col], ascending=[True] * len(group_columns) + [False, True])

        dominant_info = (
            freq.groupby(group_columns, as_index=False)
            .head(1)
            .copy()
        )

        dominant_info["dominant_ratio"] = (
            dominant_info["count"] / dominant_info["base_count_not_quote"]
        )

        dominant_info = dominant_info.rename(columns={
            en_no_col: "dominant_EN_No",
            en_label_col: "dominant_EN_label",
            "count": "dominant_count",
        })

        dominant_info = dominant_info[
            group_columns + [
                "base_count_not_quote",
                "dominant_EN_No",
                "dominant_EN_label",
                "dominant_count",
                "dominant_ratio",
            ]
        ]

        if dominance_threshold is not None:
            dominant_info["dominant_over_threshold"] = (
                dominant_info["dominant_ratio"] >= dominance_threshold
            )

    # ----------------------------
    # 3) merge everything into df_docu
    # ----------------------------
    merge_frames = [
        sen_count,
        sen_count_has_E,
        sen_count_not_quote,
        sen_count_has_E_not_quote,
        dominant_info,
    ]

    for m in merge_frames:
        cols_to_add = [c for c in m.columns if c not in df_docu.columns or c in group_columns]
        m = m[cols_to_add]
        df_docu = df_docu.merge(m, on=group_columns, how="left")

    # ----------------------------
    # 4) fill count columns with 0
    # ----------------------------
    count_cols = [
        "sen_count",
        "sen_count_has_E",
        "sen_count_not_quote",
        "sen_count_has_E_not_quote",
        "base_count_not_quote",
        "dominant_count",
    ]
    for col in count_cols:
        if col in df_docu.columns:
            df_docu[col] = df_docu[col].fillna(0)

    return df_docu

In [26]:
df_docu = add_docu_sentence_en_info(
    df_docu=df_docu,
    df_sen=df_sen,
    group_columns=["docu_id"],
    en_no_col="last_EN_No",
    en_label_col="last_EN_label",
    has_e_col="sent_has_E",
    in_quote_col="last_EN_in_quote",
    #dominance_threshold=0.6,   # optional
)

In [27]:
df_docu.columns

Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio'],
      dtype='object')

In [29]:
df_docu.to_csv(DOCU_CSV, index=False, encoding= "utf-8")
print(f"{DOCU_CSV}로 저장했습니다.")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"완료 - {stamp}")

..\csv\original_csv\세종문어_document_정보.csv로 저장했습니다.
완료 - 20260307_143235


In [1]:
#대상 파일 읽어오기.
from pathlib import Path
from datetime import datetime
import pandas as pd

stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv")
df_word = pd.read_csv(CSV_PATH)
df = df_word.copy()

print(f"*** 파일 읽기 완료: {CSV_PATH} ({len(df):,}행): {stamp} ***")
df_word.columns

*** 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv (12,973,652행): 20260322_11-21-49 ***


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id2', 'rev_word_id2',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0', 'V_label_0',
       'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form', 'EN_label',
       'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'quote_type', 'vx_len', 'Next_VX_No',
       'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order', 'V_word_id',
       'f_word_id'],
      dtype='object')

In [2]:
# ==========================
# df9.6버전에서 9.7버전으로 변경할 때, vx컬럼명 대문자로 변경
# ==========================
df = df.rename(columns={
    'word_id2': "word_id",
    'rev_word_id2': "rev_word_id",
    "V_form": "V_form_old",
    "V_label": "V_label_old",
    "V_form_0": "V_form",
    "V_label_0": "V_label",
    'V_word_id': "V0_word_id",
    'vx_len': "VX_len",
    'vx0_No': "VX0_No",
    'vx0_form': "VX0_form",
    'vx0_order': "VX0_order",
})

print(f"컬럼 변경 완료 - {stamp}")
df.columns
#..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv로 저장했습니다.
# 완료 - 20260322_11-03-56

컬럼 변경 완료 - 20260322_11-21-49


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id'],
      dtype='object')

In [4]:
df_word.columns

Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id2', 'rev_word_id2',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0', 'V_label_0',
       'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form', 'EN_label',
       'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'quote_type', 'vx_len', 'Next_VX_No',
       'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order', 'V_word_id',
       'f_word_id'],
      dtype='object')

In [5]:
# 저장하기

output_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv"
df.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
print(f"완료 - {stamp}")

..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv로 저장했습니다.
완료 - 20260322_11-24-10
